# 05 — 去重分析

## 精确去重 vs 模糊去重：互补的两步流程

> **生产级 pipeline 的标准做法（FineWeb, Nemotron-CC 都采用）**：
>
> **Step 1: 精确去重（本 Notebook A 部分）**
> - 工具：MD5/SHA256/xxhash
> - 复杂度：O(n)，极快
> - 能捕获：15-25% 的完全重复文档
> - 限制：不能捕获“几乎相同”的文档
>
> **Step 2: 模糊去重（本 Notebook B 部分）**
> - 工具：MinHash + LSH
> - 复杂度：O(n x B x K)，较慢
> - 能捕获：近似重复（Jaccard 相似度 > 閘值）
> - 限制：概率性（有 false positive/negative）
>
> **两步的顺序很重要**：先精确后模糊，精确去重能大幅减少 MinHash 的计算量。

## Cell Group A: 精确去重（Exact Deduplication）

In [ ]:
# === 环境初始化 + 加载两档 Gen1 输出数据 ===
import sys, json, random, re
sys.path.insert(0, '..')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
from pathlib import Path
from src.utils.config_loader import load_run_config, get_output_path, print_config_summary
from src.dedup.exact_dedup import exact_dedup, url_dedup, analyze_duplicate_sources, compute_doc_hash
from src.gen1.pipeline import read_jsonl

def sanitize_text(text):
    return re.sub(r'[\ud800-\udfff]', '', text)

def sanitize_docs(docs):
    for d in docs:
        if 'text' in d:
            d['text'] = sanitize_text(d['text'])
    return docs

# --- 依赖文件校验 ---
run_cfg = load_run_config()
current_mode = run_cfg.get('run_mode', 'smoke_test')
gen1_dir = get_output_path(1, run_cfg)
gen1_file = gen1_dir / 'gen1_output.jsonl'
assert gen1_file.exists(), f"缺少: {gen1_file}，请先运行 scripts/run_gen1.py"

# --- 加载两档数据 ---
dual_docs = {}
for mode in ['smoke_test', 'full_run']:
    mode_cfg = load_run_config(run_mode_override=mode)
    mode_gen1_dir = get_output_path(1, mode_cfg)
    mode_gen1_file = mode_gen1_dir / 'gen1_output.jsonl'
    if mode_gen1_file.exists():
        dual_docs[mode] = sanitize_docs(read_jsonl(mode_gen1_file))
        print(f"  {mode}: {len(dual_docs[mode]):,} 条文档")
    else:
        print(f"  {mode}: 数据不存在，跳过")

# 当前 mode 的数据（后续详细分析用）
print_config_summary(run_cfg)
docs = dual_docs.get(current_mode, dual_docs.get('smoke_test', []))
print(f"\n当前模式 ({current_mode}): {len(docs):,} 条文档")

In [2]:
# === 精确重复统计 ===
# 用 xxhash 对每篇文档计算哈希值，然后统计哈希频次，
# 识别完全相同的重复文档。输出总文档数、唯一哈希数、
# 重复组数和预期去除率，并展示 Top 5 最常重复的文档内容，
# 帮助直观了解重复的来源和模式。

# 统计重复情况
from collections import Counter

hashes = [compute_doc_hash(d['text']) for d in docs]
hash_counter = Counter(hashes)
duplicated = {h: c for h, c in hash_counter.items() if c > 1}

print(f"\ud83d\udcca \u7cbe\u786e\u91cd\u590d\u7edf\u8ba1:")
print(f"  \u603b\u6587\u6863\u6570: {len(docs):,}")
print(f"  \u552f\u4e00\u54c8\u5e0c\u6570: {len(hash_counter):,}")
print(f"  \u91cd\u590d\u7ec4\u6570: {len(duplicated):,}")
print(f"  \u91cd\u590d\u6587\u6863\u6570: {sum(c-1 for c in duplicated.values()):,}")
print(f"  \u9884\u671f\u53bb\u9664\u7387: {sum(c-1 for c in duplicated.values())/len(docs):.1%}")

# Top 5 最常重复的文档
print(f"\n  \u6700\u5e38\u91cd\u590d\u7684\u6587\u6863\uff08Top 5\uff09:")
for h, count in Counter(hashes).most_common(5):
    if count > 1:
        for d in docs:
            if compute_doc_hash(d['text']) == h:
                print(f"    [{count}\u6b21\u91cd\u590d] {d['text'][:80]!r}...")
                break

ERROR:tornado.general:Uncaught exception in ZMQStream callback
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Un

rrors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 684, in <lambda>
    self.io_loop.add_callback(lambda: self._handle_events(self.socket, 0))
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  

In [3]:
# === 执行精确去重 + 重复来源分析 ===
# 调用 exact_dedup 执行精确去重（支持文本归一化），输出去重率等统计指标。
# 然后通过 analyze_duplicate_sources 区分"同域名重复"和"跨域名重复"，
# 跨域名重复通常来自转载站/聚合站，是数据污染的重要信号。
# 输出 Top 10 重复域名，帮助识别低质量数据源。

# 执行精确去重
deduped_exact, exact_stats = exact_dedup(docs, normalize=True)
print(f"\n\u2705 \u7cbe\u786e\u53bb\u91cd\u5b8c\u6210:")
for k, v in exact_stats.items():
    if k != 'most_duplicated':
        print(f"  {k}: {v}")

# 分析重复来源
dup_sources = analyze_duplicate_sources(docs)
print(f"\n\ud83d\udcca \u91cd\u590d\u6587\u6863\u6765\u6e90\u5206\u6790:")
print(f"  \u603b\u91cd\u590d\u7ec4: {dup_sources['total_duplicate_groups']:,}")
print(f"  \u540c\u57df\u540d\u91cd\u590d: {dup_sources['same_domain_duplicates']:,}")
print(f"  \u8de8\u57df\u540d\u91cd\u590d: {dup_sources['cross_domain_duplicates']:,}")
print(f"\n  Top 10 \u91cd\u590d\u57df\u540d:")
for item in dup_sources['top_10_dup_domains'][:5]:
    print(f"    {item['domain']}: {item['count']} \u6b21")

ERROR:tornado.general:Uncaught exception in ZMQStream callback
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 185-186: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

rialize(msg, ident)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 727, in serialize
    content = self.pack(content)
              ^^^^^^^^^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 145, in orjson_packer
    return json_packer(obj)
           ^^^^^^^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 111, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 223-224: surrogates not allowed
ERROR:tornado.general:Uncaught exception in zmqstream callback
UnicodeEncodeError: 'utf-8' codec can't encode c

## Cell Group B: MinHash 模糊去重

### 核心数学原理

**Jaccard 相似度**：
$$J(A, B) = \\frac{|A \\cap B|}{|A \\cup B|}$$

**MinHash 近似**：
用 $k$ 个随机哈希函数，每个取集合的最小值。
相同最小值的比例 ≈ Jaccard 相似度。

**LSH（Locality Sensitive Hashing）**：
将 $k$ 个哈希值分成 $b$ 个 band，每个 band 有 $r = k/b$ 行。
相似度 $s$ 的两个文档，被放入同一桶的概率 ≈ $1-(1-s^r)^b$。

### 关键参数 Trade-off

| 参数 | 增大效果 | 减小效果 |
|---|---|---|
| num_hashes | 估计更准确，更慢 | 估计有噪声，更快 |
| num_buckets | 閘值更高（更精确匹配才去重） | 閘值更低（更宽松去重） |
| threshold | 只去除高相似文档 | 去除更多相似文档 |

### 去重粒度对比

| 粒度 | 方法 | 速度 | 代表 |
|---|---|---|---|
| 文档级 | 整篇文档的 MinHash | 快 | FineWeb, Nemotron-CC |
| 句子级 | 3-sentence sliding window | 慢（~5倍） | C4 |

文档级：快但可能遗漏段落级重复；句子级：更彻底但计算成本高得多。本项目使用文档级。

In [4]:
# === MinHash 原理验证 ===
# 用手工构造的三个句子验证 MinHash 的 Jaccard 相似度估算准确性：
# text_a 和 text_b 内容高度相似（仅末尾几词不同），预期 Jaccard > 0.5；
# text_a 和 text_c 内容完全无关（一个是英语句子，一个是 ML 术语），预期 Jaccard < 0.3。
# 这一步确认 MinHash 的估算方向正确，再用于实际数据。

from src.dedup.minhash_dedup import MinHashLSH, compute_minhash, estimate_jaccard

# 参数设置
minhash = MinHashLSH(
    num_hashes=128,
    num_buckets=8,      # rows_per_band = 128/8 = 16
    threshold=0.8,      # 80% Jaccard 相似度视为重复
    shingle_n=5,
)

# 人工验证 MinHash 的准确性
text_a = "The quick brown fox jumps over the lazy dog. A classic English pangram."
text_b = "The quick brown fox jumps over the lazy dog. A classic sentence in English."
text_c = "Completely different content about machine learning and neural networks."

sig_a = compute_minhash(text_a)
sig_b = compute_minhash(text_b)
sig_c = compute_minhash(text_c)

print("MinHash \u9a8c\u8bc1\uff08Jaccard \u76f8\u4f3c\u5ea6\u4f30\u7b97\uff09:")
print(f"  \u76f8\u4f3c\u53e5\u5b50 A vs B: {estimate_jaccard(sig_a, sig_b):.3f} (\u9884\u671f > 0.5)")
print(f"  \u4e0d\u540c\u53e5\u5b50 A vs C: {estimate_jaccard(sig_a, sig_c):.3f} (\u9884\u671f < 0.3)")

MinHash 验证（Jaccard 相似度估算）:


  相似句子 A vs B: 0.719 (预期 > 0.5)
  不同句子 A vs C: 0.000 (预期 < 0.3)


In [5]:
# === 在实际数据上执行 MinHash 模糊去重 ===
# 对精确去重后的数据执行 MinHash LSH 模糊去重，
# 参数：128 个哈希函数、8 个 band（rows_per_band=16）、0.8 Jaccard 阈值。
# 这一步捕获"几乎相同但不完全一致"的近似重复文档（如转载时微修改的内容）。
# 最后输出两步去重（精确 + MinHash）的综合结果：原始文档数、各步去除数、总去除率。

# 在实际数据上运行 MinHash 去重
print(f"\n对 {len(deduped_exact):,} 条文档（精确去重后）进行 MinHash 去重...")
deduped_minhash, minhash_stats = minhash.dedup(deduped_exact)
deduped_minhash = sanitize_docs(deduped_minhash)  # 防止 surrogate 字符导致 UnicodeEncodeError

print(f"\n📊 MinHash 去重结果:")
for k, v in minhash_stats.items():
    if k != 'parameters':
        print(f"  {k}: {v}")

# 两步去重汇总
total_removed = len(docs) - len(deduped_minhash)
print(f"\n📊 两步去重汇总:")
print(f"  原始文档: {len(docs):,}")
print(f"  精确去重后: {len(deduped_exact):,} (-{len(docs)-len(deduped_exact):,})")
print(f"  MinHash去重后: {len(deduped_minhash):,} (-{len(deduped_exact)-len(deduped_minhash):,})")
print(f"  总去除率: {total_removed/len(docs):.1%}")


对 401 条文档（精确去重后）进行 MinHash 去重...
  🔄 MinHash 去重: 401 条文档
     num_hashes=128, num_buckets=8, threshold=0.8
  建立 MinHash LSH 索引...


  MinHash 签名计算:   0%|          | 0/401 [00:00<?, ?it/s]

  MinHash 签名计算:   0%|          | 2/401 [00:00<02:13,  2.99it/s]

  MinHash 签名计算:   1%|          | 3/401 [00:01<02:14,  2.97it/s]

  MinHash 签名计算:   1%|          | 4/401 [00:01<01:40,  3.93it/s]

...

  MinHash 签名计算: 100%|█████████▉| 399/401 [01:21<00:00,  6.29it/s]

  MinHash 签名计算: 100%|█████████▉| 400/401 [01:21<00:00,  6.10it/s]

  MinHash 签名计算: 100%|██████████| 401/401 [01:21<00:00,  6.53it/s]

  MinHash 签名计算: 100%|██████████| 401/401 [01:21<00:00,  4.90it/s]

  查找候选重复对...
  找到 9 对相似文档 (Jaccard >= 0.8)
  ✅ MinHash 去重: 401 → 393 条 | 去除 8 条近似重复 (2.0%)

📊 MinHash 去重结果:
  total_input: 401
  unique_kept: 393
  near_duplicates_removed: 8
  dedup_rate: 0.02
  candidate_pairs: 9
  jaccard_threshold: 0.8

📊 两步去重汇总:
  原始文档: 409
  精确去重后: 401 (-8)
  MinHash去重后: 393 (-8)
  总去除率: 3.9%


In [6]:
# === 去重对数据质量的影响分析 ===
# 对比去重前后的 N-gram 多样性变化（unigram/bigram/trigram unique ratio），
# 预期去重后多样性提升：移除重复内容后，剩余文档的词汇分布更加丰富均匀。
# 多样性提升说明去重有效移除了冗余内容，而非误删了有价值的独特文档。

# 去重对质量的影响分析
print("\ud83d\udcca \u53bb\u91cd\u5bf9\u8d28\u91cf\u5206\u5e03\u7684\u5f71\u54cd:")

from src.evaluation.diversity_metrics import compute_all_ngram_diversities

# 去重前后 N-gram 多样性对比
orig_diversity = compute_all_ngram_diversities([d['text'] for d in docs[:200]])
dedup_diversity = compute_all_ngram_diversities([d['text'] for d in deduped_minhash[:200]])

print(f"\n  N-gram \u591a\u6837\u6027\u5bf9\u6bd4\uff08\u53bb\u91cd\u524d vs \u540e\uff09:")
for ng in ['unigram', 'bigram', 'trigram']:
    orig_val = orig_diversity.get(ng, {}).get('unique_ratio', 0)
    dedup_val = dedup_diversity.get(ng, {}).get('unique_ratio', 0)
    change = dedup_val - orig_val
    arrow = '\u2191' if change > 0 else '\u2193'
    print(f"  {ng:<10}: {orig_val:.4f} \u2192 {dedup_val:.4f}  {arrow}{abs(change):.4f}")


  N-gram 多样性对比（去重前 vs 后）:
  unigram   : 0.1721 → 0.1733  ↑0.0012
  bigram    : 0.6613 → 0.6683  ↑0.0070
  trigram   : 0.8861 → 0.8964  ↑0.0103


ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x10c111660>>)
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 103, in json_packer
    

).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/ipykernel/iostream.py", line 715, in _flush
    self.session.send(
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/fineweb-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 856, in send
    to_send = self.serialize(msg, ident)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pengjuzhao/Desktop/claude code/tiktok-ml-projects/finew

## Cell Group D: 跨 Dump 去重讨论（理论分析，不运行代码）

> **跨 Dump 去重的概念**
>
> Common Crawl 每年爬取 4-6 次（每次叫一个 dump），同一个 URL 可能在不同 dump 中被重复爬取。
>
> FineWeb 处理了 96 个 dump，面临两个层次的跨 dump 去重挑战：
> 1. **URL 级别去重**：同一 URL 在不同 dump 出现 → 只保留最新版本
> 2. **内容级别去重**：不同 URL 但内容相同（转载、聚合站）
>
> **本项目的简化**：我们只用 1 个 WARC 文件，不涉及跨 dump 问题。
>
> **FineWeb-2 的“再水化（Rehydration）”策略**：
> 去重后，根据原始重复次数做加权上采样：
> - 重复 2 次 → 保留 2 份（质量高的内容应该多看一次）
> - 重复 10+ 次 → 保留 1 份（避免过拟合）
> - 重复 1000+ 次 → 保留 1 份（明确的垃圾内容）
>
> 这是一个在“去除噪声”和“保留重要内容”之间精细权衡的策略。

In [7]:
# === 去重最终结论 + 双模式对比 ===
from src.dedup.minhash_dedup import MinHashLSH as MinHashLSH2

print("=" * 70)
print("  去重分析 — 双模式对比")
print("=" * 70)

print(f"\n{'档位':<14} {'输入文档':<10} {'精确去重后':<12} {'精确去重率':<12} {'MinHash后':<10} {'MinHash率':<12} {'综合去重率':<10}")
print("-" * 90)

for mode in ['smoke_test', 'full_run']:
    if mode not in dual_docs:
        continue
    mode_docs = dual_docs[mode]

    # 精确去重
    deduped_e, stats_e = exact_dedup(mode_docs, normalize=True)
    deduped_e = sanitize_docs(deduped_e)  # 防止 surrogate 字符
    # MinHash 去重
    mh = MinHashLSH2(num_hashes=128, num_buckets=8, threshold=0.8, shingle_n=5)
    deduped_m, stats_m = mh.dedup(deduped_e)
    deduped_m = sanitize_docs(deduped_m)  # 防止 surrogate 字符
    total_rem = len(mode_docs) - len(deduped_m)
    total_rate = total_rem / len(mode_docs) if mode_docs else 0

    marker = " ◀" if mode == current_mode else ""
    print(f"{mode:<14} {len(mode_docs):<10,} {len(deduped_e):<12,} "
          f"{stats_e.get('dedup_rate', 0):<12.1%} {len(deduped_m):<10,} "
          f"{stats_m.get('dedup_rate', 0):<12.1%} {total_rate:<10.1%}{marker}")

print()
print("  当前模式详细结果（上方各 Cell 的分析基于此模式）:")
print(f"  原始文档: {len(docs):,}")
print(f"  精确去重率: {exact_stats.get('dedup_rate', 0):.1%}")
print(f"  MinHash 去重率: {minhash_stats.get('dedup_rate', 0):.1%}")
print(f"  最终保留: {len(deduped_minhash):,} 条")
total_removed = len(docs) - len(deduped_minhash)
print(f"  综合去重率: {total_removed/len(docs):.1%}")
print()
print("  结论：去重是清洗流程不可缺少的一步，")
print("  能在不影响质量的前提下减少重复 token，")
print("  提升训练效率（更快收敛，避免过拟合重复内容）。")

  去重分析 — 双模式对比

档位             输入文档       精确去重后        精确去重率        MinHash后   MinHash率     综合去重率     
------------------------------------------------------------------------------------------
  🔄 精确去重: 409 → 401 条 | 去除 8 条 (2.0%)
  🔄 MinHash 去重: 401 条文档
     num_hashes=128, num_buckets=8, threshold=0.8
  建立 MinHash LSH 索引...


  MinHash 签名计算:   0%|          | 0/401 [00:00<?, ?it/s]

  MinHash 签名计算:   0%|          | 2/401 [00:00<02:02,  3.25it/s]

  MinHash 签名计算:   1%|          | 3/401 [00:00<02:07,  3.11it/s]

  MinHash 签名计算:   1%|          | 4/401 [00:01<01:41,  3.93it/s]

...

  MinHash 签名计算: 100%|█████████▉| 3081/3084 [11:09<00:00,  3.38it/s]

  MinHash 签名计算: 100%|█████████▉| 3083/3084 [11:10<00:00,  4.39it/s]

  MinHash 签名计算: 100%|██████████| 3084/3084 [11:10<00:00,  3.33it/s]

  MinHash 签名计算: 100%|██████████| 3084/3084 [11:10<00:00,  4.60it/s]

  查找候选重复对...
  找到 244 对相似文档 (Jaccard >= 0.8)
  ✅ MinHash 去重: 3,084 → 2,967 条 | 去除 117 条近似重复 (3.8%)
full_run       3,242      3,084        4.9%         2,967      3.8%         8.5%      

  当前模式详细结果（上方各 Cell 的分析基于此模式）:
  原始文档: 409
  精确去重率: 2.0%
  MinHash 去重率: 2.0%
  最终保留: 393 条
  综合去重率: 3.9%

  结论：去重是清洗流程不可缺少的一步，
  能在不影响质量的前提下减少重复 token，
  提升训练效率（更快收敛，避免过拟合重复内容）。
